# CCGA — conic construction theory

The *wedge of points* ladder and how a conic is built in CCGA ($\mathbb{R}^{5,3}$):

`point(1) → dipole(2) → tripole(3) → quadpole(4) → pentapole(5) → conic(7 / 1)`

Companion to `tests/test_conic_construction.py`. Everything below is verified
numerically with `kingdon`. The two questions driving this notebook:

1. **What differentiates the bare 5-point wedge from the grade-7 conic `… ∧ Iod`?**
2. **How is conic type characterized, and what happens when ideal points enter the build?**

In [ ]:
import numpy as np
from ccga.algebra import Iod, Iinfd, I_inv, print_null, format_null
from ccga.point import point, point_at_infinity
from ccga.objects import (make_point_pair, make_conic_tripole, make_conic_quadpole,
                          make_conic_pentapole, pentapole_to_conic, make_ideal_point)
from ccga.operations import dual, grades, is_zero
from ccga.classify import (ipns_to_coeffs, conic_subtype, conic_discriminant,
                           conic_type, asymptotic_directions, classify)

def wedge(pts):
    R = pts[0]
    for p in pts[1:]: R = R ^ p
    return R

def coeffs(pts):
    return ipns_to_coeffs(dual(wedge(pts) ^ Iod))

## 1. The construction ladder

Each extra point raises the grade by one. The grade-5 *pentapole* is the last
pure wedge-of-points rung; wedging `Iod` (grade 2) lands the grade-7 conic.

In [ ]:
pts = [point(0,0), point(3,0), point(0,2), point(2,2), point(-1,1)]
ladder = {
    'point     (1)': pts[0],
    'dipole    (2)': wedge(pts[:2]),
    'tripole   (3)': wedge(pts[:3]),
    'quadpole  (4)': wedge(pts[:4]),
    'pentapole (5)': wedge(pts[:5]),
    'conic     (7)': wedge(pts[:5]) ^ Iod,
}
for name, mv in ladder.items():
    print(f'{name}   grade {grades(mv)}')

## 2. Finding 1 — the pentapole already IS the conic (same locus)

A conic is *one linear relation* among the six Veronese coordinates
$(1,x,y,\tfrac{x^2}2,\tfrac{y^2}2,xy)$, so all of its points share a 5-D subspace of
$V_6$. Five generic points span it — hence `q ∧ P5 = 0 ⟺ q on the conic`,
exactly like the grade-7 conic. `∧ Iod` adds nothing to incidence.

In [ ]:
P5 = wedge(pts); C7 = P5 ^ Iod
A,B,C,D,E,F = ipns_to_coeffs(dual(C7))

def on_conic(x):
    r = np.roots([B, C*x+E, A*x*x+D*x+F])
    return [point(x, float(t.real)) for t in r if abs(t.imag) < 1e-9]

on  = [q for x in np.linspace(-2,4,9) for q in on_conic(x)]
off = [point(1,1), point(0.5,0.5), point(5,5)]
print('on-conic  : q^P5==0 all?', all(is_zero(q^P5) for q in on),
      ' q^C7==0 all?', all(is_zero(q^C7) for q in on))
print('off-conic : q^P5!=0 all?', all(not is_zero(q^P5) for q in off),
      ' q^C7!=0 all?', all(not is_zero(q^C7) for q in off))

## 3. Finding 2 — the dual is what `Iod` changes

| | OPNS grade | dual | dual grade |
|---|---|---|---|
| `P5` | 5 | $-\tfrac12\,(s\wedge I_\infty^{\triangleright})$ | 3 |
| `C7 = P5 ∧ Iod` | 7 | $s$ (clean conic vector) | 1 |

`Iod = eobar ∧ eo3` supplies the two directions points can't reach (W₂), collapsing
the pseudoscalar dual to grade 1 instead of the `Iinfd`-smeared grade 3.

In [ ]:
s = dual(C7)
print('dual(C7) grade', grades(s), '-> clean conic vector:')
print('  ', format_null(s, tol=1e-6))
print('dual(P5) grade', grades(dual(P5)), '-> s smeared by Iinfd:')
print('  ', format_null(dual(P5), tol=1e-6))
print()
# verify dual(P5) = -1/2 (s ^ Iinfd) across random conics
for seed in range(5):
    rng = np.random.default_rng(seed)
    rp = [point(*(rng.standard_normal(2)*2)) for _ in range(5)]
    Q5 = wedge(rp); sv = dual(Q5 ^ Iod)
    print(f'seed {seed}: dual(P5) == -1/2 (s ^ Iinfd)?',
          is_zero(dual(Q5) - (-0.5)*(sv ^ Iinfd)))

## 4. Findings 3 & 4 — conic type from ideal points

Conic type = incidence with the line at infinity, $\Delta = C^2 - 4AB$.
Constructively, set by how many **true Veronese ideal points**
`point_at_infinity(vx,vy)` $\in I_\infty$ enter the build:

- **0** → ellipse ($\Delta<0$)
- **2** → hyperbola ($\Delta>0$); the two directions are its **asymptotes**
- **1 double** (merging limit) → parabola ($\Delta\to0$)

In [ ]:
def show(name, pts):
    A,B,C,D,E,F = coeffs(pts)
    print(f'{name:38s} Delta={conic_discriminant(A,B,C):+9.3f}  type={conic_subtype(A,B,C,D,E,F)}')

show('0 ideal (5 finite)', pts)
show('2 ideal: vinf(1,0)+vinf(0,1)', [point(0,0),point(2,0),point(0,2),
                                       point_at_infinity(1,0), point_at_infinity(0,1)])
show('2 ideal: vinf(2,1)+vinf(1,-1)', [point(1,0),point(2,1),point(0,3),
                                        point_at_infinity(2,1), point_at_infinity(1,-1)])

In [ ]:
# asymptotic directions ARE the ideal points used
H = make_conic_pentapole(point(1,0),point(2,1),point(0,3),
                         point_at_infinity(2,1), point_at_infinity(1,-1))[0]
print('hyperbola asymptotes:', [tuple(round(c,3) for c in d) for d in asymptotic_directions(H)])
print('ellipse  asymptotes:', asymptotic_directions(wedge(pts)))
print()
# parabola as the merging limit of two ideal points
fin = [point(0,0), point(2,0.5), point(1,3)]
for eps in (0.3, 0.05, 0.005):
    A,B,C,_,_,_ = coeffs(fin + [point_at_infinity(1,eps), point_at_infinity(1,-eps)])
    print(f'eps={eps:6.3f}  Delta={conic_discriminant(A,B,C):+.3e}  type={conic_subtype(A,B,C,0,0,0)}')

## 5. Finding 5 — `point_at_infinity` ≠ `make_ideal_point`

The genuine point at infinity is the *pure-quadratic* Veronese limit
`point_at_infinity(v) = (vx²/2)einf1 + (vy²/2)einf2 + (vx·vy)einf3`.
`make_ideal_point(v)` (a CGA round point with `eo` dropped) keeps a linear `e1/e2`
part — it lies on a *different* conic and does **not** control type/asymptotes.

In [ ]:
v1, v2 = (1,1), (1,-1)
fin = [point(0,0), point(2,0), point(0,3)]
s_vinf  = dual(wedge(fin+[point_at_infinity(*v1), point_at_infinity(*v2)]) ^ Iod)
s_ideal = dual(wedge(fin+[make_ideal_point(*v1),  make_ideal_point(*v2)])  ^ Iod)
for name, s in [('point_at_infinity', s_vinf), ('make_ideal_point', s_ideal)]:
    A,B,C,*_ = ipns_to_coeffs(s)
    print(f'{name:18s} A={A:+.2f} B={B:+.2f} C={C:+.2f}'
          f'   asym test (1,1)={A+C+B:+.2e}  (1,-1)={A-C+B:+.2e}')

## 6. Conic ∨ conic intersection — the grade-6 object

Two conics meet in **4 points** (Bézout). The regressive product is a grade-6 blade

$$I_4 = C_1 \vee C_2 \;\propto\; p_1\wedge p_2\wedge p_3\wedge p_4\wedge I_o^{\triangleright} = Q\wedge I_o^{\triangleright},$$

the quadpole of the 4 points gauge-fixed by `Iod` (same `Iod` as the grade-7 conic).
Incidence: `q ∧ I4 = 0 ⟺ q` is an intersection point. Recover the quadpole with
`Q = (einf3 ∧ einfbar) | I4`, then `extract_quadpole`. The 4 points can be **real**,
an **imaginary** conjugate pair, or **ideal** (at infinity).

In [ ]:
from ccga.operations import meet
from ccga.extract import (conic_intersection, intersection_quadpole,
                          intersection_points, intersection_reality, extract_quadpole)
from ccga.objects import make_ellipse

# pencil: two conics through the same 4 points
q1,q2,q3,q4 = point(0,0), point(3,0), point(0,2), point(2,3)
K1 = wedge([q1,q2,q3,q4, point(-1,1), Iod])
K2 = wedge([q1,q2,q3,q4, point(4,1),  Iod])
M = conic_intersection(K1, K2)
print('meet grade:', grades(M))
print('q ∧ I4 = 0 for the 4 shared points?', all(is_zero(p ^ M) for p in (q1,q2,q3,q4)))
Q = intersection_quadpole(M)
print('recovered points:', sorted((round(x,3),round(y,3)) for x,y in extract_quadpole(Q)))

### Reality of the 4 intersection points (Bézout split)

`intersection_reality` returns `{real, imaginary, ideal}` summing to 4. Note two circles
always share the **two imaginary circular points at infinity**, so they meet in 2 finite +
2 ideal.

In [ ]:
cases = [
    ('4 real (ellipses 3x2, 2x3)', make_ellipse(3,2)[0], make_ellipse(2,3)[0]),
    ('2 real (overlapping circles)', make_ellipse(2,2,0,0)[0], make_ellipse(2,2,2,0)[0]),
    ('0 real (disjoint circles)', make_ellipse(1,1,0,0)[0], make_ellipse(1,1,5,0)[0]),
    ('ideal (hyperbolas, shared asymptote)',
        wedge([point(0,0),point(2,1),point(1,3), point_at_infinity(1,0), point_at_infinity(0,1), Iod]),
        wedge([point(0,0),point(2,1),point(1,3), point_at_infinity(1,0), point_at_infinity(1,1), Iod])),
]
for name, A, B in cases:
    r = intersection_reality(A, B)
    print(f'{name:38s} {r}  (sum {sum(r.values())})')
print()
print('real finite points (4-real case):',
      sorted((round(x,3),round(y,3)) for x,y in intersection_points(make_ellipse(3,2)[0], make_ellipse(2,3)[0])))

## Summary

- The bare grade-5 **pentapole** is the conic as a point-set; `∧ Iod` is a pure
  gauge/dual fix giving the clean grade-1 IPNS conic ($\operatorname{dual}(P_5)=-\tfrac12\,s\wedge I_\infty^{\triangleright}$).
- Conic **type** is its incidence with the line at infinity ($\Delta=C^2-4AB$),
  set constructively by the number of true ideal points (`point_at_infinity`) in the
  build: 0 → ellipse, 1 double → parabola, 2 → hyperbola (= asymptotic directions).
- Use `point_at_infinity`, **not** `make_ideal_point`, for the conic-type theory.